# Setup
## Imports

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
from generative_social_choice.utils.postprocessing import (
    collect_results_and_plot,
    plot_sorted_utility_distributions,
    plot_candidate_distribution_stacked,
    clustered_barplot_with_error_bars,
    plot_scalar_clustered_confidence_intervals,
    save_figure,
    ResultConfig,
)


## Load data

In [ ]:

METHOD = "exact"
CONFIDENCE_LEVEL = .95
N_BOOTSTRAP = 400
# Figure format when saving: "png" or "pdf"
FIG_FORMAT = "pdf"
# Set to a path (e.g. "figures") to save the line plot from collect_results_and_plot; None to not save
SAVE_DIR = "figures/"

result_configs = [
    ResultConfig(
        #name="Ours (LLM Embeddings)",
        name="Ours",
        embedding_type="llm",
        run_ids=list(range(10)),
    ),
    #ResultConfig(
    #    name="Ours (Fish Embeddings)",
    #    embedding_type="fish",
    #    run_ids=[f"fish_{i}" for i in range(10)],
    #),
    #ResultConfig(
    #    name="Baseline (LLM Embeddings)",
    #    embedding_type="seed_statement",
    #    run_ids=range(15),
    #    pipeline="fish",
    #),
    ResultConfig(
        name="Baseline",
        embedding_type="llm",
        run_ids=range(15),
        pipeline="fish",
        # Keep earlier rounds' candidates in the pool, as described by Fish et al.
        fish_candidate_pool="persistent",
    )
]

results = collect_results_and_plot(
    result_configs, method=METHOD, confidence_level=CONFIDENCE_LEVEL, n_bootstrap=N_BOOTSTRAP,
    save_dir=SAVE_DIR, fig_format=FIG_FORMAT, figsize=(9, 5),
    # Two-line labels for "GenerativeCC (ours)", so the legend stays clear of the curves
    legend_break_longer_than=30,
)

results["scalar_confidence_intervals"]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

utility_metrics = results["scalar_confidence_intervals"].drop(columns=['Gini', 'Mean Log', '2*Mean Log'])
log_metric = results["scalar_confidence_intervals"][['Mean Log']]
gini_metric = results["scalar_confidence_intervals"][['Gini']]

# Column-width figure, drawn at its printed size (\columnwidth = 3.3 in), so the 7 pt fonts are the printed
# size. Lines, markers and spacings are scaled down to match, and the subplots are placed by hand to fit
# them in one column.
FIG_WIDTH_IN, FIG_HEIGHT_IN = 3.3, 2.28
FONT_SIZE = 7
# Width of the data area of the two single-metric subplots, and the gap between a subplot and the
# y-axis labels of the next one (inches). The three-cluster subplot gets the remaining width.
NARROW_SUBPLOT_WIDTH_IN = 0.36
SUBPLOT_GAP_IN = 0.1
with plt.rc_context({
    "lines.linewidth": 0.9,
    "axes.linewidth": 0.5,
    "grid.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,
    "ytick.major.size": 2,
    "xtick.major.pad": 2,
    "ytick.major.pad": 1.5,
    "axes.labelpad": 1.5,
}):
    scalar_confidence_intervals_plot = plot_scalar_clustered_confidence_intervals(
        utility_metrics,
        y_label="Utility",
        legend_loc="lower left",
        fig_size=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
        secondary_axis_df=log_metric,
        secondary_y_label="log(Utility)",
        tertiary_axis_df=gini_metric,
        tertiary_y_label="Gini",
        cluster_width=0.6,
        font_size=FONT_SIZE,
        marker_size=3.5,
        cap_size=2.3,
        cap_thick=0.6,
        # "GenerativeCC\n(ours)", so the legend clears the Minimum cluster
        legend_break_longer_than=12,
    )
    fig = scalar_confidence_intervals_plot
    utility_ax, log_ax, gini_ax = fig.axes
    log_ax.yaxis.set_major_locator(MultipleLocator(0.01))
    gini_ax.yaxis.set_major_locator(MultipleLocator(0.005))
    # Slanted metric names, so they don't collide in the narrow column
    for ax in fig.axes:
        ax.tick_params(axis="x", labelrotation=30)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment("right")
            label.set_rotation_mode("anchor")
    # Compact legend in the empty lower left corner, below the Mean and Bottom 50% clusters
    utility_ax.legend(loc="lower left", fontsize=FONT_SIZE, handletextpad=0.2, borderpad=0.3,
                      labelspacing=0.3, borderaxespad=0.3, handlelength=1.2)

    # Place the subplots in a row, each one just right of the y-axis labels of the next
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    def y_axis_label_width(ax):
        return (ax.get_window_extent(renderer).x0 - ax.yaxis.get_tightbbox(renderer).x0) / fig.dpi
    def x_axis_label_height(ax):
        return (ax.get_window_extent(renderer).y0 - ax.xaxis.get_tightbbox(renderer).y0) / fig.dpi
    label_widths = [y_axis_label_width(ax) for ax in fig.axes]
    pad = 0.03
    wide_subplot_width = (FIG_WIDTH_IN - 2 * pad - sum(label_widths) - 2 * SUBPLOT_GAP_IN
                          - 2 * NARROW_SUBPLOT_WIDTH_IN)
    bottom = pad + max(x_axis_label_height(ax) for ax in fig.axes)
    top = FIG_HEIGHT_IN - 0.06  # room for the top tick labels, which stick out above the axes
    x = pad
    for ax, label_width, width in zip(fig.axes, label_widths,
                                      [wide_subplot_width, NARROW_SUBPLOT_WIDTH_IN, NARROW_SUBPLOT_WIDTH_IN]):
        x += label_width
        ax.set_position([x / FIG_WIDTH_IN, bottom / FIG_HEIGHT_IN, width / FIG_WIDTH_IN,
                         (top - bottom) / FIG_HEIGHT_IN])
        x += width + SUBPLOT_GAP_IN
if SAVE_DIR:
    save_figure(scalar_confidence_intervals_plot, Path(SAVE_DIR) / "scalar_confidence_intervals", FIG_FORMAT)

In [ ]:

METHOD = "exact"
CONFIDENCE_LEVEL = .95
N_BOOTSTRAP = 400
# Figure format when saving: "png" or "pdf"
FIG_FORMAT = "pdf"
# Set to a path (e.g. "figures") to save the line plot from collect_results_and_plot; None to not save
SAVE_DIR = "figures/"

result_configs = [
    ResultConfig(
        name="Ours (LLM Embs.)",
        #name="Ours",
        embedding_type="llm",
        run_ids=list(range(10)),
    ),
    ResultConfig(
        name="Ours (Their Embs.)",
        embedding_type="fish",
        run_ids=[f"fish_{i}" for i in range(10)],
    ),
    ResultConfig(
        name="Baseline (LLM Embs.)",
        embedding_type="seed_statement",
        run_ids=range(15),
        pipeline="fish",
    ),
    ResultConfig(
        name="Baseline (Their Embs.)",
        embedding_type="llm",
        run_ids=range(15),
        pipeline="fish",
    )
]

results = collect_results_and_plot(
    result_configs, method=METHOD, confidence_level=CONFIDENCE_LEVEL, n_bootstrap=N_BOOTSTRAP,
    save_dir=SAVE_DIR, fig_format=FIG_FORMAT, figsize=(10, 6),
    # Two-line labels for "GenerativeCC (ours, ...)", so the legend stays clear of the curves
    legend_break_longer_than=42,
)

results["scalar_confidence_intervals"]

In [ ]:
import matplotlib.pyplot as plt

# Only variants of the same pipeline are comparable here: both baseline series use the round-only
# candidate pool (see the embedding ablation appendix), so the baseline (a) and GenerativeCC (b) get
# separate axes, stacked in one column-width figure.
cis = results["scalar_confidence_intervals"]
panels = [
    ("(a) Baseline", ["Baseline (LLM Embs.)", "Baseline (Their Embs.)"]),
    ("(b) GenerativeCC", ["Ours (LLM Embs.)", "Ours (Their Embs.)"]),
]
# Shorter tick labels to fit the column width
short_metric_names = {"Mean of\nBottom 50%": "Bottom\n50%", "Minimum": "Min"}

embedding_ablation_fig = plt.figure(figsize=(4.4, 6.0), layout="constrained")
for subfig, (title, names) in zip(embedding_ablation_fig.subfigures(len(panels), 1), panels):
    panel_cis = cis.loc[cis.index.get_level_values(0).isin(names)].rename(columns=short_metric_names)
    plot_scalar_clustered_confidence_intervals(
        panel_cis.drop(columns=['Gini', 'Mean Log', '2*Mean Log']),
        y_label="Utility",
        legend_loc="outside upper center",
        legend_ncol=2,
        legend_frameon=False,
        secondary_axis_df=panel_cis[['Mean Log']],
        secondary_y_label="log(Utility)",
        tertiary_axis_df=panel_cis[['Gini']],
        tertiary_y_label="Gini",
        # Narrower single-metric subplots than the default of one cluster's width
        width_ratios=[3, 0.6, 0.6],
        font_size=10,
        fig=subfig,
    )
    legend = subfig.legends[0]
    legend.set_title(title, prop={"size": 10, "weight": "bold"})
    # The title names the pipeline, so keep only the embeddings in the labels, e.g. "LLM Embs." out of
    # "GenerativeCC (ours, LLM Embs.)" (the full series names still set the colors, matching Fig 6)
    for text in legend.get_texts():
        text.set_text(text.get_text().split(" (", 1)[1].removesuffix(")").removeprefix("ours, "))
if SAVE_DIR:
    save_figure(embedding_ablation_fig, Path(SAVE_DIR) / "scalar_confidence_intervals_all_stacked", FIG_FORMAT)
    save_figure(embedding_ablation_fig, Path(SAVE_DIR) / "scalar_confidence_intervals_all_stacked", "png")

In [ ]:
scalar_confidence_intervals_plot = clustered_barplot_with_error_bars(
    results["scalar_confidence_intervals"].iloc[:,:3],
    y_label="Agreement",
    legend_loc="center left",
    fig_size=(4.5, 4),
    secondary_axis_df=results["scalar_confidence_intervals"].iloc[:,[3]],
    secondary_y_label="Gini"
)

## Statement Ablation

In [ ]:
import json
from pathlib import Path
import pandas as pd

from generative_social_choice.utils.helper_functions import get_results_paths
from generative_social_choice.ratings.utility_matrix import get_baseline_generate_slate_results
from generative_social_choice.utils.postprocessing import (
    scalar_utility_metrics,
    bootstrap_df_rows,
    plot_scalar_clustered_confidence_intervals,
    clustered_barplot_with_error_bars,
)


def get_ablation_results_for_run(
    run_id: str,
    ablation_name: str,
    embedding_type: str = "llm",
    generation_model: str = "4o",
    labelling_model: str = "4o-mini",
):
    rp = get_results_paths(
        labelling_model=labelling_model,
        baseline=False,
        embedding_type=embedding_type,
        generation_model=generation_model,
        run_id=run_id,
    )
    assignments_dir = (
        rp["results_dir"] / "ablations" / ablation_name / f"{labelling_model}_for_labelling" / "assignments"
    )

    files = {p.stem: p for p in assignments_dir.glob("*.json")}
    utilities = pd.DataFrame(columns=list(files.keys()))
    algo_assignments = pd.DataFrame(columns=list(files.keys()))

    for algo_name, file_path in files.items():
        data = json.loads(file_path.read_text())
        utilities[algo_name] = pd.Series(data["utilities"], index=data["agent_ids"])
        algo_assignments[algo_name] = pd.Series(data["assignments"], index=data["agent_ids"])

    return utilities, algo_assignments

In [ ]:
METHOD = "exact"
CONFIDENCE_LEVEL = 0.95
N_BOOTSTRAP = 400
ABLATION_NAME = "llm_subsample_3"

# Use same run IDs for ablation and ours-llm comparison.
ablation_runs = [str(i) for i in range(10)]
seed_statement_ids = {f"s{i}" for i in range(1, 7)}


def _load_slate(assignment_file: Path) -> list[str]:
    data = json.loads(assignment_file.read_text())
    return data["slate"]


# 1) Build ablation utility series and verify no seed statements in slates.
ablation_series = []
for rid in ablation_runs:
    rp = get_results_paths(
        labelling_model="4o-mini",
        baseline=False,
        embedding_type="llm",
        generation_model="4o",
        run_id=rid,
    )
    ablation_assignment_file = (
        rp["results_dir"]
        / "ablations"
        / ABLATION_NAME
        / "4o-mini_for_labelling"
        / "assignments"
        / f"{METHOD}.json"
    )
    ablation_slate = _load_slate(ablation_assignment_file)
    assert not (set(ablation_slate) & seed_statement_ids), (
        f"Seed statements found in ablation slate for run {rid}: "
        f"{set(ablation_slate) & seed_statement_ids}"
    )

    u_df, _ = get_ablation_results_for_run(
        run_id=rid,
        ablation_name=ABLATION_NAME,
        embedding_type="llm",
        generation_model="4o",
        labelling_model="4o-mini",
    )
    ablation_series.append(u_df[METHOD])

# 2) Ours (LLM Embeddings): reuse loaded results and verify no seed statements.
ours_llm_series = []
for rid in ablation_runs:
    rp = get_results_paths(
        labelling_model="4o-mini",
        baseline=False,
        embedding_type="llm",
        generation_model="4o",
        run_id=rid,
    )
    ours_assignment_file = rp["assignments"] / f"{METHOD}.json"
    ours_slate = _load_slate(ours_assignment_file)
    assert not (set(ours_slate) & seed_statement_ids), (
        f"Seed statements found in ours-llm slate for run {rid}: "
        f"{set(ours_slate) & seed_statement_ids}"
    )

    ours_llm_series.append(results["utility_df_dict"]["Ours (LLM Embs.)"][int(rid)][METHOD])

# 3) Baseline: the same runs as in the main results (cell 4), not the round-only runs of the
# embedding ablation (cell 6). This pipeline returns one utility series per run.
baseline_utilities, _ = get_baseline_generate_slate_results(
    run_ids=range(15), embedding_type="llm", candidate_pool="persistent"
)
baseline_fish_series = [baseline_utilities[col] for col in baseline_utilities.columns]

utilities_multidf = pd.DataFrame({
    **{
        ("Ours (30 statements)", i): ablation_series[i]
        for i in range(len(ablation_series))
    },
    **{
        ("Ours (42 statements)", i): ours_llm_series[i]
        for i in range(len(ours_llm_series))
    },
    **{
        ("Baseline (30 statements)", i): baseline_fish_series[i]
        for i in range(len(baseline_fish_series))
    },
})
utilities_multidf.columns = pd.MultiIndex.from_tuples(utilities_multidf.columns)

scalar_metrics_per_run = scalar_utility_metrics(utilities_multidf)
scalar_confidence_intervals = bootstrap_df_rows(
    scalar_metrics_per_run,
    confidence_level=CONFIDENCE_LEVEL,
    n_bootstrap=N_BOOTSTRAP,
)

from IPython.display import display
import matplotlib.pyplot as plt

utility_metrics = scalar_confidence_intervals.drop(columns=['Gini', 'Mean Log', '2*Mean Log'])
log_metric = scalar_confidence_intervals[['Mean Log']]
gini_metric = scalar_confidence_intervals[['Gini']]

ablation_comparison_fig = plot_scalar_clustered_confidence_intervals(
    utility_metrics,
    y_label="Utility",
    # Below the Mean and Bottom 50% clusters, clear of the data
    legend_loc="lower left",
    legend_break_longer_than=20,
    fig_size=(11, 5),
    secondary_axis_df=log_metric,
    secondary_y_label="log(Utility)",
    tertiary_axis_df=gini_metric,
    tertiary_y_label="Gini",
)

display(ablation_comparison_fig)

# Save as PDF
ablation_comparison_fig.savefig("statement_ablation_comparison.pdf", bbox_inches="tight")

# Prevent duplicate automatic rendering in notebook output
plt.close(ablation_comparison_fig)

### Older Plots

In [ ]:
fig = plot_sorted_utility_distributions(results["utility_df_dict"]["Ours (LLM Embeddings)"][0])
if SAVE_DIR:
    save_figure(fig, Path(SAVE_DIR) / "sorted_utility_distributions", FIG_FORMAT)

In [ ]:
fig = plot_candidate_distribution_stacked(results["algo_assignments"]["Ours (LLM Embeddings)"][0])